# Medical Devices API -> pandas DataFrame
Open this notebook from the cloned project in a Databricks Git folder. All data is held in driver memory; no Volume, wheel upload, Spark or table is required. The result is `devices_df`, a pandas DataFrame.

In [ ]:
%pip install "requests>=2.32.3,<3" "pandas>=2.2,<3"

Restart Python after installation if requested by the runtime. include_details=True fetches all detail rows once per registration. The notebook defaults to 4 worker threads with a shared limit of 2 request starts per second, including retries. Each worker has its own Session. HTTP 429/5xx cooldown is shared; access errors stop the run. At about 30,000 registrations, the rate-limit floor alone is about 4.2 hours; network latency, retries and server limits add time. This rate is a client setting, not an approved server quota. Set detail_workers=1 for the original sequential mode (delay_seconds applies there), or include_details=False for listing fields only.

In [ ]:
from pathlib import Path
import logging
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/moh_medical_devices').is_dir()), None)
if project_root is not None:
    sys.path.insert(0, str(project_root / 'src'))

try:
    from moh_medical_devices.client import Client, Config
    from moh_medical_devices.dataframe import fetch_dataframe
except ModuleNotFoundError as exc:
    raise RuntimeError('Open this notebook from the cloned project Git folder, or pip install the project first.') from exc

logging.basicConfig(level=logging.INFO)
page_size = 1000
include_details = True
search = ''
declaration_only = False
delay_seconds = 2
detail_workers = 4
requests_per_second = 2

In [ ]:
# Clear a previous result before a rerun so failure cannot leave stale data here.
devices_df = None
client = Client(Config(delay_seconds=delay_seconds))
try:
    devices_df = fetch_dataframe(
        client, page_size=page_size, include_details=include_details,
        search=search, declaration_only=declaration_only,
        detail_workers=detail_workers, requests_per_second=requests_per_second,
    )
finally:
    client.close()

print(devices_df.attrs['extraction'])
display(devices_df.head(20))

## Result
`devices_df` retains each source listing row, including duplicate registration numbers, normalized columns, all original fields and raw_json. `details` is a list of all rows returned by the detail endpoint; details_raw_json retains the complete detail response. The detail fetch time is recorded separately. No files or tables are written. This DataFrame lives only in the current Python session; an interrupted run must restart from the beginning. The API has no verified snapshot token, so same-count changes outside the first page cannot be ruled out. An API/detail failure raises an exception rather than returning partial success.